# Stability Audit for Logistic Regression

You trained a logistic regression. It scores well. But would a different equally-good model
give the same predictions? The same feature importances? The same decisions for individual patients?

This notebook walks through a complete audit using **rashomon-py** on the Breast Cancer dataset.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from rashomon import RashomonSet

## 1. Fit a logistic regression

We use 10 geometric features from the Breast Cancer dataset. The model achieves
strong accuracy -- nothing here is caused by a "bad" model.

In [ ]:
data = load_breast_cancer()
feature_names = ['radius', 'texture', 'perimeter', 'area', 'smoothness',
                 'compactness', 'concavity', 'concave_pts', 'symmetry', 'fractal_dim']

X = StandardScaler().fit_transform(data.data[:, :10])
y = data.target.astype(float)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

rs = RashomonSet(
    estimator="logistic",
    epsilon=0.03,               # 3% loss tolerance
    epsilon_mode="percent_loss",
    sampler="hitandrun",        # asymptotically exact membership sampling
    random_state=42,
    C=0.5,
    safety_override=True,
).fit(X_train, y_train)

print(f"Train accuracy: {rs.score(X_train, y_train):.1%}")
print(f"Test accuracy:  {rs.score(X_test, y_test):.1%}")
print(f"Epsilon:        {rs.epsilon_:.4f}")

## 2. How many patients get an unstable diagnosis?

**Ambiguity** counts the fraction of patients where some equally-good model
flips the predicted label.

In [ ]:
amb = rs.ambiguity(X_test, threshold_mode="fixed", threshold_value=0.5)

print(f"Ambiguous: {amb['n_ambiguous']}/{len(X_test)} ({amb['ambiguity_rate']:.1%})")
print(f"\nThese patients get a different diagnosis depending on which")
print(f"equally-good model the clinician happens to use.")

In [ ]:
rs.plot_ambiguity(X_test)

Red points are patients whose prediction interval crosses the decision boundary.
Their bars show the range of predictions across equally-good models.

**Discrepancy** measures worst-case disagreement between any two models in the set:

In [ ]:
disc = rs.discrepancy(X_test, n_samples=200, n_pairs=200, random_state=42)
print(f"Max pair disagreement: {disc['max_pair_disagreement']:.1%}")

## 3. How stable are the coefficients?

The **Variable Importance Cloud (VIC)** shows how each coefficient varies
across near-optimal models. Wide spread means the feature's role is not
pinned down by the data.

In [ ]:
vic = rs.variable_importance_cloud(n_samples=300, random_state=42)

print("Coefficient ranges across the Rashomon set:")
print(f"{'Feature':<14} {'Min':>8} {'Mean':>8} {'Max':>8} {'Std':>8}")
print("-" * 48)
for i, name in enumerate(feature_names):
    print(f"{name:<14} {vic['min'][i]:>8.3f} {vic['mean'][i]:>8.3f} "
          f"{vic['max'][i]:>8.3f} {vic['std'][i]:>8.3f}")

In [ ]:
rs.plot_vic(n_samples=300, feature_names=feature_names, random_state=42)

## 4. Certificate vs Hit-and-Run comparison

Certificates use an ellipsoidal approximation (fast, closed-form).
Hit-and-Run samples from the true Rashomon set (slower, asymptotically exact).
Let's compare them.

In [ ]:
# Certificate-based intervals (milliseconds)
cert_intervals = rs.coef_intervals()

# Compare widths
print(f"{'Feature':<14} {'Cert width':>12} {'VIC width':>12} {'Ratio':>8}")
print("-" * 50)
for i, name in enumerate(feature_names):
    cert_w = cert_intervals[i, 1] - cert_intervals[i, 0]
    vic_w = vic['max'][i] - vic['min'][i]
    ratio = cert_w / vic_w if vic_w > 0 else float('inf')
    print(f"{name:<14} {cert_w:>12.4f} {vic_w:>12.4f} {ratio:>7.1f}x")

At d=10, certificates track the Hit-and-Run intervals within about 1.5x.
This makes them a reliable fast screen at low dimensionality.

## 5. Bootstrap CIs vs Rashomon intervals

Bootstrap CIs measure *sampling uncertainty*. Rashomon intervals measure
*model multiplicity* -- a fundamentally different quantity.

In [ ]:
comp = rs.compare_to_bootstrap(
    X_train, y_train,
    n_bootstrap=500, n_rashomon=500,
    confidence=0.90,
    feature_names=feature_names,
    random_state=42,
)

print(f"{'Feature':<14} {'Boot width':>12} {'VIC width':>12} {'Ratio':>8}")
print("-" * 50)
for row in comp['comparison']:
    print(f"{row['feature']:<14} {row['bootstrap_width']:>12.4f} "
          f"{row['rashomon_width']:>12.4f} {row['width_ratio']:>7.1f}x")

The Rashomon intervals are 4-11x wider than bootstrap CIs. These are not
two estimates of the same thing. Bootstrap says "how uncertain is the
best-fit?" The Rashomon set says "how many qualitatively different models
perform nearly as well?"

## 6. Interpreting the results

**What we found:**
- ~23% of test patients get a diagnosis that depends on which equally-good model is used
- Two near-optimal models can disagree on ~8% of patients
- Every feature's coefficient varies substantially across the Rashomon set
- Bootstrap CIs dramatically understate the range of viable model explanations

**What this means for a practitioner:**
- If you're deploying this model for clinical decisions, ~1 in 5 patients
  falls in a zone where the prediction is not determined by the data alone
- Standard model evaluation (accuracy, bootstrap CIs) would not reveal this
- These patients may need additional scrutiny, a second opinion, or flagging

**What this does NOT mean:**
- It does not mean the model is bad -- accuracy is high
- It does not mean we should pick a different model class
- It means the data admits multiple valid explanations, and we should be
  transparent about which predictions are robust and which are not